# Data Exploration — 黄金 & 医药生物 因子分析

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 1. 加载原始数据
from src.data_fetcher.akshare_source import AKShareSource
from src.data_fetcher.fred_source import FREDSource

ak = AKShareSource()
fred = FREDSource()

data = {}
data.update(ak.fetch_all('2020-01-01'))
data.update(fred.fetch_all('2020-01-01'))
# 用黄金概念指数替代 yfinance 数据做演示
data['gold_futures'] = data.get('gold_concept_cn', pd.DataFrame())

print(f'Loaded {len(data)} data sources:')
for k, v in data.items():
    if not v.empty:
        print(f'  {k}: {len(v)} rows, {v.date.min().date()} ~ {v.date.max().date()}')

In [ ]:
# 2. 目标变量 — 黄金和医药板块的周度收益率分布
from src.factors.factor_pipeline import FactorPipeline

pipeline = FactorPipeline()
factors_df = pipeline.compute_all(data, verbose=False)
print(f'Factor matrix: {factors_df.shape}')

In [ ]:
# 3. 目标收益率走势
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# SW Medical close price
med = data['sw_medical'].set_index('date')['close'].sort_index()
med_w = med.resample('W-FRI').last()
axes[0, 0].plot(med_w.index, med_w.values)
axes[0, 0].set_title('SW Medical Index Weekly Close')

# Medical weekly returns
med_ret = med_w.pct_change() * 100
axes[0, 1].hist(med_ret.dropna(), bins=50, alpha=0.7, color='steelblue')
axes[0, 1].axvline(0, color='red', linestyle='--')
axes[0, 1].set_title(f'Medical Weekly Returns Distribution\nMean={med_ret.mean():.2f}%, Std={med_ret.std():.2f}%')

# Gold concept close price
gold = data['gold_concept_cn'].set_index('date')['close'].sort_index()
gold_w = gold.resample('W-FRI').last()
axes[1, 0].plot(gold_w.index, gold_w.values, color='goldenrod')
axes[1, 0].set_title('Gold Concept Index Weekly Close')

# Gold weekly returns
gold_ret = gold_w.pct_change() * 100
axes[1, 1].hist(gold_ret.dropna(), bins=50, alpha=0.7, color='goldenrod')
axes[1, 1].axvline(0, color='red', linestyle='--')
axes[1, 1].set_title(f'Gold Weekly Returns Distribution\nMean={gold_ret.mean():.2f}%, Std={gold_ret.std():.2f}%')

plt.tight_layout()
plt.show()

In [ ]:
# 4. 因子与目标的相关性热力图（前 15 个因子）
from src.factors.factor_pipeline import align_factors_with_target

med_price = data['sw_medical'].set_index('date')['close'].sort_index()
X_med, y_med = align_factors_with_target(factors_df, med_price, shift_target=1)
print(f'Aligned medical data: X={X_med.shape}, y={y_med.shape}')

# 合并后看相关性
combined = pd.concat([X_med.iloc[:, :15], y_med['magnitude']], axis=1)
combined.columns = list(X_med.columns[:15]) + ['next_return']
corr = combined.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Factor-Target Correlation Matrix (Medical, Top 15 Factors)')
plt.tight_layout()
plt.show()

In [ ]:
# 5. IC 分析 — 哪个因子最有效？
from src.features.selector import ic_analysis

ic_df = ic_analysis(X_med, y_med['magnitude'], method='spearman')

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['steelblue' if v > 0 else 'coral' for v in ic_df['IC']]
ic_sorted = ic_df.dropna(subset=['IC']).sort_values('IC')
ax.barh(range(len(ic_sorted)), ic_sorted['IC'].values, color=colors)
ax.set_yticks(range(len(ic_sorted)))
ax.set_yticklabels(ic_sorted['factor'].values)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Rank IC (Spearman)')
ax.set_title('Factor Information Coefficient — Medical Sector')
plt.tight_layout()
plt.show()

In [ ]:
# 6. 共线性诊断
from src.features.selector import vif_scores, correlation_matrix

# 高相关因子对
high_corr = correlation_matrix(X_med, threshold=0.85)
print(f'Factor pairs with |corr| > 0.85: {len(high_corr)}')
if len(high_corr) > 0:
    display(high_corr.head(10))

# VIF 最高的因子
try:
    vif = vif_scores(X_med.iloc[:, :20].dropna())
    print(f'\nTop 5 factors by VIF:')
    display(vif.head())
except Exception as e:
    print(f'VIF computation skipped: {e}')

In [ ]:
# 7. 时间序列切分可视化
from src.features.engineer import split_timeseries
splits = split_timeseries(X_med, y_med[['magnitude']], test_ratio=0.2)

print(f"Train: {splits['X_train'].index[0].date()} ~ {splits['X_train'].index[-1].date()} ({len(splits['X_train'])} weeks)")
print(f"Test:  {splits['X_test'].index[0].date()} ~ {splits['X_test'].index[-1].date()} ({len(splits['X_test'])} weeks)")

fig, ax = plt.subplots(figsize=(14, 4))
ax.axvspan(splits['X_train'].index[-1], splits['X_test'].index[-1], alpha=0.15, color='orange', label='Test Set')
ax.axvspan(splits['X_train'].index[0], splits['X_train'].index[-1], alpha=0.15, color='steelblue', label='Train Set')
ax.plot(y_med.index, y_med['magnitude'].values, linewidth=0.8, color='gray')
ax.set_title('Medical Weekly Returns — Train/Test Split')
ax.set_ylabel('Weekly Return (%)')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

以上探索完成：
1. 数据加载和概览
2. 目标分布分析
3. 因子-目标相关性
4. IC 分析
5. 共线性诊断
6. 时间序列切分可视化